In [0]:
%sql
from pyspark.sql.functions import *

gold_table = "payment_app.gold.dim_date"

if not spark.catalog.tableExists(gold_table):

    date_df = spark.sql("""
    SELECT explode(
        sequence(
            to_date('2025-01-01'),
            to_date('2030-12-31'),
            interval 1 day
        )
    ) AS full_date
    """)

    dim_date = (
        date_df
        .withColumn("date_key", date_format(col("full_date"), "yyyyMMdd").cast("int"))
        .withColumn("day", dayofmonth(col("full_date")))
        .withColumn("month", month(col("full_date")))
        .withColumn("quarter", quarter(col("full_date")))
        .withColumn("year", year(col("full_date")))
        .withColumn("weekday", date_format(col("full_date"), "EEEE"))
        .select(
            "date_key",
            "full_date",
            "day",
            "month",
            "quarter",
            "year",
            "weekday"
        )
    )

    dim_date.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(gold_table)

print("dim_date completed")